# Data Cleaning

In [13]:
import json
import gzip
from pathlib import Path
from typing import Any
import glob

import pandas as pd

In [14]:
good_json = """
{
  "company_id": 123,
  "country": "PL",
  "features": {
    "assets": 45000.0,
    "net_profit": -1200.5
  },
  "bankrupt": 1,
  "tags": ["manufacturing", "export"],
  "note": null
}
"""

In [15]:
obj=json.loads(good_json)
obj

{'company_id': 123,
 'country': 'PL',
 'features': {'assets': 45000.0, 'net_profit': -1200.5},
 'bankrupt': 1,
 'tags': ['manufacturing', 'export'],
 'note': None}

In [16]:
type(obj)

dict

In [17]:
obj["features"]

{'assets': 45000.0, 'net_profit': -1200.5}

In [19]:
data_dir=Path("../data")
paths=sorted(data_dir.glob("*.json*"))
paths

[PosixPath('../data/poland-bankruptcy-data-2009-mvp-features.json.gz'),
 PosixPath('../data/poland-bankruptcy-data-2009.json.gz'),
 PosixPath('../data/taiwan-bankruptcy-data-test-features.json.gz'),
 PosixPath('../data/taiwan-bankruptcy-data.json.gz')]

In [23]:
poland_paths=[path for path in paths if "poland" in path.name]
taiwan_path=[path for path in paths if "taiwan" in path.name]

In [32]:
poland_paths[0]

PosixPath('../data/poland-bankruptcy-data-2009-mvp-features.json.gz')

In [25]:
def load_json(path:str | Path) -> Any:
    p=Path(path)
    if p.suffix == '.gz':
        with gzip.open(p,'rt',encoding='utf-8') as f:
            return json.load(f)
        
    with p.open(encoding='utf-8') as f:
        return json.load(f)

In [37]:
path_1=paths[0]
payload0=load_json(path_1)
payload0.keys()

dict_keys(['schema', 'data', 'metadata'])

In [31]:
def extract_records(payload:Any) -> list[dict [str,Any]]:
    if not isinstance(payload,dict):
        raise TypeError("Expected the Json payload to be a dict")
    if 'data' in payload:
        records=payload["data"]
    elif 'observations' in payload:
        records=payload["observations"]
    else:
        raise KeyError(
            "Could not find records. expected key 'data' or 'observayions "
            f"Found keys: {list(payload.keys())}"
        )
        
    if not isinstance(records,list):
        raise TypeError("Expected record container to be list")
    if records and not isinstance(records[0],dict):
        raise TypeError("Expected records to be a list of dicts")
    
    return records

In [42]:
records0=extract_records(payload0)
records0

[{'company_id': 4,
  'feat_1': 0.024526,
  'feat_2': 0.43236,
  'feat_3': 0.27546,
  'feat_4': 1.7833,
  'feat_5': -10.105,
  'feat_6': 0.56944,
  'feat_7': 0.024526,
  'feat_8': 1.3057,
  'feat_9': 1.0509,
  'feat_10': 0.56453,
  'feat_11': 0.024526,
  'feat_12': 0.069747,
  'feat_13': 0.036812,
  'feat_14': 0.024526,
  'feat_15': 3299.4,
  'feat_16': 0.11063,
  'feat_17': 2.3129,
  'feat_18': 0.024526,
  'feat_19': 0.018876,
  'feat_20': 86.317,
  'feat_21': 0.62795,
  'feat_22': 0.055446,
  'feat_23': 0.018876,
  'feat_24': 0.56944,
  'feat_25': 0.56453,
  'feat_26': 0.11063,
  'feat_27': 0.44844,
  'feat_28': 0.73869,
  'feat_29': 4.26,
  'feat_30': 0.28624,
  'feat_31': 0.018876,
  'feat_32': 103.81,
  'feat_33': 3.5161,
  'feat_34': 0.12824,
  'feat_35': 0.055446,
  'feat_36': 1.3068,
  'feat_37': 3.9624,
  'feat_38': 0.64524,
  'feat_39': 0.042673,
  'feat_40': 0.17826,
  'feat_41': 0.1805,
  'feat_42': 0.042673,
  'feat_43': 158.55,
  'feat_44': 72.237,
  'feat_45': 0.079819,
 

In [43]:
def record_to_frame(records:list[dict[str,Any]]) -> pd.DataFrame:
    has_nested=any(
        isinstance(v,(dict,list))
        for r in records
        for v in r.values()
    )
    if has_nested:
        return pd.DataFrame.from_records(records)
    return pd.json_normalize(records)

In [51]:
df0=record_to_frame(records0)
df0

,company_id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_55,feat_56,feat_57,feat_58,feat_59,feat_60,feat_61,feat_62,feat_63,feat_64
0,4,0.024526,0.43236,0.275460,1.78330,-10.1050,0.569440,0.024526,1.30570,1.05090,...,5012.600,0.048398,0.043445,0.95160,0.142980,4.2286,5.0528,98.783,3.6950,3.48440
1,32,0.266780,0.41576,0.620140,3.89080,64.8330,0.000000,0.332720,1.40520,1.71680,...,17461.000,0.199130,0.456630,0.81067,0.000000,4.6673,3.9767,45.608,8.0030,10.38400
2,34,0.003892,0.85020,0.271460,1.37260,10.3600,0.000000,0.003892,0.17620,2.09600,...,99.183,0.003116,0.025980,0.99815,0.812060,10.2010,4.0764,126.870,2.8770,NaN
3,36,0.044384,1.06430,-0.529800,0.43939,-158.4000,-0.525530,0.044384,-0.14114,1.07410,...,-5644.000,0.068995,-0.295480,0.93100,-0.793930,21.5290,7.8248,203.050,1.7975,2.90500
4,40,0.013141,0.38626,0.380470,2.24300,44.2000,0.000000,0.013141,1.58890,0.88321,...,3220.000,-0.132720,0.021412,0.98720,0.000000,3.3917,4.1778,126.500,2.8855,2.81780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,10420,-0.154710,0.54835,-0.092151,0.77832,-88.4950,0.000000,-0.154710,0.82365,0.68069,...,-587.000,-0.434960,-0.342540,1.16480,0.071182,5.0058,9.6251,222.910,1.6375,1.00630
522,10423,-0.063118,0.22214,0.046678,1.78630,-19.9790,-0.146300,-0.063118,3.50090,0.13055,...,242.490,-0.735500,-0.081163,1.37710,0.209440,2.2131,3.1494,165.970,2.1992,0.14604
523,10424,-0.292150,0.43423,0.178540,2.70870,-2.1142,0.000000,-0.292150,1.30290,1.17020,...,178.050,0.171590,-0.516380,0.78637,0.000000,7.1604,30.4140,32.590,11.2000,1.63220
524,10441,-0.302630,0.83198,0.091339,1.20530,-25.3720,0.006386,-0.302630,0.20195,1.45550,...,5289.000,0.019910,-1.801200,0.94720,1.730900,6.9923,4.7323,111.580,3.2713,3.13850


In [52]:
df0.columns

Index(['company_id', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5',
       'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12',
       'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18',
       'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24',
       'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30',
       'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36',
       'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42',
       'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48',
       'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54',
       'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60',
       'feat_61', 'feat_62', 'feat_63', 'feat_64'],
      dtype='str')

In [59]:
def normalize_colunms(columns:list[str])-> list[str]:
    out: list[str]=[]
    for c in columns:
        c=str(c)
        c=c.strip()
        c=c.lower()
        c=c.replace(" ",'_')
        c=c.replace("-","_")
        c="".join(ch for ch in c if ch.isalnum() or ch=="_")
        while "__" in c:
            c=c.replace("__","_")
        out.append(c)
    return out

In [60]:
df_temp0=df0.copy()
df_temp0.columns=normalize_colunms([str(c) for c in df_temp0.columns])

In [61]:
df_temp0.columns

Index(['company_id', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5',
       'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12',
       'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18',
       'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24',
       'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30',
       'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36',
       'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42',
       'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48',
       'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54',
       'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60',
       'feat_61', 'feat_62', 'feat_63', 'feat_64'],
      dtype='str')

In [62]:
def clean_financials(df:pd.DataFrame)-> pd.DataFrame:
    df2=df.copy()
    df2.columns=normalize_colunms([str(c) for c in df2.columns])
    for col in df2.columns:
        df2[col]=pd.to_numeric(df2[col],errors="coerce")
    return df2

In [63]:
df_clean0=clean_financials(df0)

In [64]:
df_clean0.info()

<class 'pandas.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 65 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   company_id  526 non-null    int64  
 1   feat_1      526 non-null    float64
 2   feat_2      526 non-null    float64
 3   feat_3      526 non-null    float64
 4   feat_4      525 non-null    float64
 5   feat_5      526 non-null    float64
 6   feat_6      526 non-null    float64
 7   feat_7      526 non-null    float64
 8   feat_8      525 non-null    float64
 9   feat_9      526 non-null    float64
 10  feat_10     526 non-null    float64
 11  feat_11     526 non-null    float64
 12  feat_12     525 non-null    float64
 13  feat_13     525 non-null    float64
 14  feat_14     526 non-null    float64
 15  feat_15     525 non-null    float64
 16  feat_16     525 non-null    float64
 17  feat_17     525 non-null    float64
 18  feat_18     526 non-null    float64
 19  feat_19     525 non-null    float64
 20 

In [65]:
def wrangle(path : str | Path) -> pd.DataFrame:
    payload=load_json(path)
    records=extract_records(payload)
    df_raw=record_to_frame(records)
    return clean_financials(df_raw)

In [66]:
wrangle(path_1)

,company_id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_55,feat_56,feat_57,feat_58,feat_59,feat_60,feat_61,feat_62,feat_63,feat_64
0,4,0.024526,0.43236,0.275460,1.78330,-10.1050,0.569440,0.024526,1.30570,1.05090,...,5012.600,0.048398,0.043445,0.95160,0.142980,4.2286,5.0528,98.783,3.6950,3.48440
1,32,0.266780,0.41576,0.620140,3.89080,64.8330,0.000000,0.332720,1.40520,1.71680,...,17461.000,0.199130,0.456630,0.81067,0.000000,4.6673,3.9767,45.608,8.0030,10.38400
2,34,0.003892,0.85020,0.271460,1.37260,10.3600,0.000000,0.003892,0.17620,2.09600,...,99.183,0.003116,0.025980,0.99815,0.812060,10.2010,4.0764,126.870,2.8770,NaN
3,36,0.044384,1.06430,-0.529800,0.43939,-158.4000,-0.525530,0.044384,-0.14114,1.07410,...,-5644.000,0.068995,-0.295480,0.93100,-0.793930,21.5290,7.8248,203.050,1.7975,2.90500
4,40,0.013141,0.38626,0.380470,2.24300,44.2000,0.000000,0.013141,1.58890,0.88321,...,3220.000,-0.132720,0.021412,0.98720,0.000000,3.3917,4.1778,126.500,2.8855,2.81780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,10420,-0.154710,0.54835,-0.092151,0.77832,-88.4950,0.000000,-0.154710,0.82365,0.68069,...,-587.000,-0.434960,-0.342540,1.16480,0.071182,5.0058,9.6251,222.910,1.6375,1.00630
522,10423,-0.063118,0.22214,0.046678,1.78630,-19.9790,-0.146300,-0.063118,3.50090,0.13055,...,242.490,-0.735500,-0.081163,1.37710,0.209440,2.2131,3.1494,165.970,2.1992,0.14604
523,10424,-0.292150,0.43423,0.178540,2.70870,-2.1142,0.000000,-0.292150,1.30290,1.17020,...,178.050,0.171590,-0.516380,0.78637,0.000000,7.1604,30.4140,32.590,11.2000,1.63220
524,10441,-0.302630,0.83198,0.091339,1.20530,-25.3720,0.006386,-0.302630,0.20195,1.45550,...,5289.000,0.019910,-1.801200,0.94720,1.730900,6.9923,4.7323,111.580,3.2713,3.13850


In [67]:
datasets_path:dict[str,Path]={
    "poland_features" : next(p for p in poland_paths if "features" in p.name.lower()),
     "poland_full" : next(p for p in poland_paths if "features" not in p.name.lower()),
      "taiwan_features" : next(p for p in taiwan_path if "features" in p.name.lower()),
          "taiwan_full" : next(p for p in taiwan_path if "features" not in p.name.lower())
}

In [68]:
datasets_path

{'poland_features': PosixPath('../data/poland-bankruptcy-data-2009-mvp-features.json.gz'),
 'poland_full': PosixPath('../data/poland-bankruptcy-data-2009.json.gz'),
 'taiwan_features': PosixPath('../data/taiwan-bankruptcy-data-test-features.json.gz'),
 'taiwan_full': PosixPath('../data/taiwan-bankruptcy-data.json.gz')}

In [69]:
frames:dict[str,pd.DataFrame]={
    name:wrangle(path) for name ,path in datasets_path.items()
}
{
    k:v.shape for k,v in frames.items()
}

{'poland_features': (526, 65),
 'poland_full': (9977, 66),
 'taiwan_features': (682, 96),
 'taiwan_full': (6137, 97)}

In [ ]:
def imbalance_ratio(y:pd.Series)->float:
    s=y.dropna().astype("int")
    counts=s.value_counts()
    if counts.shape[0] <2:
        return float("nan")
    majority=max(counts)
    minority=min(counts)
    
    if minority == 0.0:
        return float("inf")
    return majority/minority